# PlantVillage: pretrained models, SOTA-style protocol, background-robust training

**What this notebook does**
1. Downloads the *full* PlantVillage dataset (54,305 colour images, 38 classes) with its segmented images from GitHub.
2. Makes ONE fixed stratified split (80/10/10, seed 42) that every model uses, so all comparisons are on the same data.
3. Fine-tunes ImageNet-pretrained models (EfficientNet-B0, MobileNetV3-L, ResNet-50, ConvNeXt-Tiny, ViT-Small) at 224 px.
4. Reports accuracy, macro precision/recall/F1/specificity, ROC-AUC, ECE, confusion matrices and curves (CO3).
5. Compares all models on the same split (CO4) and runs the innovation experiment (CO5): background-randomisation training
   versus plain fine-tuning, 3 seeds, with a background-swap robustness test and Grad-CAM leaf-focus.

**Before you run:** Runtime -> Change runtime type -> **GPU (T4)**. Then Runtime -> Run all.
Expected time on a free T4: roughly 1.5-3 hours for everything (set `CFG["seeds"]` / `compare_models` smaller to shorten).
Results are saved in `OUT`; set `USE_DRIVE = True` in the config cell to keep them on Google Drive if the session may disconnect.

**Honesty notes**
- Numbers are produced only by running this notebook; none are pre-filled.
- PlantVillage contains several photos of the same physical leaf and a random per-image split can put near-duplicates in train and test,
  so very high accuracies (99 %+) are partly an artefact of the dataset. Published numbers use many different protocols.
  Only compare with papers that state their split, and say so in your report.

In [ ]:
import importlib, subprocess, sys
for pkg in ["timm", "opencv-python-headless", "scikit-learn", "pandas", "matplotlib"]:
    mod = {"opencv-python-headless": "cv2", "scikit-learn": "sklearn"}.get(pkg, pkg)
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, json, time, math, random, shutil, glob, queue, threading
from multiprocessing.pool import ThreadPool
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import cv2
import timm
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score, classification_report

print("torch", torch.__version__, "| timm", timm.__version__, "| cuda:", torch.cuda.is_available(),
      (torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""))

## Configuration

In [ ]:
SMOKE = os.environ.get("SMOKE") == "1"          # tiny CPU test mode (leave off in Colab)
USE_DRIVE = False                                # True: store results on Google Drive (asks you to authorise)
dev = "cuda" if torch.cuda.is_available() else "cpu"

CFG = dict(
    img=64 if SMOKE else 224,                    # input resolution (224 = standard ImageNet size)
    batch=16 if SMOKE else 64,
    epochs=1 if SMOKE else 8,                    # fine-tuning epochs
    lr=1e-3, wd=1e-2, label_smoothing=0.0,       # AdamW + OneCycle schedule
    p_bg=0.5,                                    # probability that a training image gets a random background (innovation)
    split=(0.8, 0.1, 0.1), split_seed=42,
    cap_per_class=10 if SMOKE else None,          # None = FULL dataset
    seeds=[0] if SMOKE else [0, 1, 2],           # seeds for the main-model ablation
    compare_models=(["mobilenetv3_large_100", "efficientnet_b0"] if SMOKE else
                    ["efficientnet_b0", "mobilenetv3_large_100", "resnet50", "convnext_tiny", "vit_small_patch16_224"]),
    main_model="efficientnet_b0",
    pretrained=not SMOKE,                        # ImageNet weights via timm (downloaded from Hugging Face hub)
    n_xai=16 if SMOKE else 500,                  # test images used for Grad-CAM leaf-focus
)
WORK = os.environ.get("WORK") or ("/content/work" if os.path.isdir("/content") else "./work")
OUT = os.environ.get("OUT") or os.path.join(WORK, "results")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/plantvillage_results"
for d in [WORK, OUT, os.path.join(OUT, "ckpt"), os.path.join(OUT, "runs")]:
    os.makedirs(d, exist_ok=True)
print(CFG); print("results ->", OUT)

## 1. Data: download, pair colour/segmented images, split

In [ ]:
def sh(cmd):
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)

DATA = os.environ.get("PV_ROOT") or os.path.join(WORK, "PlantVillage-Dataset")
if not os.path.isdir(os.path.join(DATA, "raw", "color")):
    sh(["git", "clone", "--depth", "1", "https://github.com/spMohanty/PlantVillage-Dataset", DATA])
try:
    print("dataset commit:", subprocess.run(["git", "-C", DATA, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
except Exception:
    pass
RAW = os.path.join(DATA, "raw")

def build_index(raw, cap, seed=42):
    """List (colour_path, segmented_path, label). Pair on the filename; fall back to the part after the last '___'
    only when it is unique on both sides (some classes carry a UUID prefix on the segmented side only)."""
    rng = np.random.RandomState(seed)
    classes = sorted(os.listdir(os.path.join(raw, "color")))
    items, counts, unpaired = [], {}, {}
    for ci, c in enumerate(classes):
        files = sorted(os.listdir(os.path.join(raw, "color", c)))
        counts[c] = len(files)
        seg_dir = os.path.join(raw, "segmented", c)
        seg_files = set(os.listdir(seg_dir))
        suf = lambda n: n.split("___")[-1]
        seg_by, col_by = {}, {}
        for sf in seg_files:
            seg_by.setdefault(suf(sf[:-len("_final_masked.jpg")]), []).append(sf)
        for f in files:
            col_by.setdefault(suf(f.rsplit(".", 1)[0]), []).append(f)
        paired = []
        for f in files:
            exact = f.rsplit(".", 1)[0] + "_final_masked.jpg"
            if exact in seg_files:
                paired.append((f, exact)); continue
            k = suf(f.rsplit(".", 1)[0])
            if len(col_by[k]) == 1 and len(seg_by.get(k, [])) == 1:
                paired.append((f, seg_by[k][0]))
        assert len({s for _, s in paired}) == len(paired), f"segmented file paired twice in {c}"
        unpaired[c] = len(files) - len(paired)
        if cap and len(paired) > cap:
            paired = [paired[j] for j in rng.choice(len(paired), cap, replace=False)]
        for f, sf in paired:
            items.append((os.path.join(raw, "color", c, f), os.path.join(seg_dir, sf), ci))
    return classes, items, counts, {c: n for c, n in unpaired.items() if n}

classes, items, full_counts, unpaired = build_index(RAW, CFG["cap_per_class"])
NC = len(classes)
print(f"{len(items)} images, {NC} classes; colour images without a segmented partner (skipped): {unpaired or 'none'}")

y_all = np.array([i[2] for i in items])
idx = np.arange(len(y_all))
tr, rest = train_test_split(idx, test_size=1 - CFG["split"][0], stratify=y_all, random_state=CFG["split_seed"])
va, te = train_test_split(rest, test_size=CFG["split"][2] / (CFG["split"][1] + CFG["split"][2]),
                          stratify=y_all[rest], random_state=CFG["split_seed"])
split_all = np.empty(len(y_all), dtype="<U5"); split_all[tr], split_all[va], split_all[te] = "train", "val", "test"

dist = pd.DataFrame({"class": classes, "full_dataset_images": [full_counts[c] for c in classes]})
for s in ["train", "val", "test"]:
    dist[s] = [int(((y_all == i) & (split_all == s)).sum()) for i in range(NC)]
dist.to_csv(f"{OUT}/dataset_distribution.csv", index=False)
sp = pd.DataFrame({"split": ["train", "val", "test"], "images": [int((split_all == s).sum()) for s in ["train", "val", "test"]]})
sp["percent"] = (100 * sp.images / sp.images.sum()).round(1)
sp.to_csv(f"{OUT}/split_distribution.csv", index=False)
print(sp.to_string(index=False))
print("class imbalance (full dataset): min", dist.full_dataset_images.min(), "max", dist.full_dataset_images.max(),
      "ratio %.1f:1" % (dist.full_dataset_images.max() / dist.full_dataset_images.min()))
dist.head(10)

## 2. Cache resized images + leaf masks (uint8 memmap on local disk)
Masks come from the dataset's segmented images and are used **only** for training-time background randomisation
and for the explanation/robustness metrics. The classifier never sees a mask.

In [ ]:
S = CFG["img"]
CACHE = os.path.join(WORK, f"cache_{S}_{CFG['cap_per_class'] or 'full'}")

def load_pair(a):
    cp, sp_, size = a
    img = Image.open(cp).convert("RGB").resize((size, size), Image.BILINEAR)
    seg = np.array(Image.open(sp_).convert("RGB"))
    m = (seg.max(axis=2) > 20).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))   # close small holes (dark lesions)
    m = cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST)
    return np.asarray(img), m

if not os.path.exists(CACHE + "_meta.npz"):
    t0 = time.time()
    X = np.lib.format.open_memmap(CACHE + "_X.npy", mode="w+", dtype=np.uint8, shape=(len(items), S, S, 3))
    M = np.lib.format.open_memmap(CACHE + "_M.npy", mode="w+", dtype=np.uint8, shape=(len(items), S, S))
    with ThreadPool(max(2, os.cpu_count() or 2)) as pool:
        for i, (img, m) in enumerate(pool.imap(load_pair, [(a, b, S) for a, b, _ in items], chunksize=32)):
            X[i], M[i] = img, m
            if i % 5000 == 0: print(f"  cached {i}/{len(items)}  ({time.time()-t0:.0f}s)", flush=True)
    X.flush(); M.flush(); del X, M
    np.savez(CACHE + "_meta.npz", y=y_all, split=split_all, classes=np.array(classes))
    print(f"cache built in {time.time()-t0:.0f}s")

class Store:
    def __init__(self, path):
        self.X = np.load(path + "_X.npy", mmap_mode="r"); self.M = np.load(path + "_M.npy", mmap_mode="r")
        meta = np.load(path + "_meta.npz")
        self.y, self.split, self.classes = meta["y"], meta["split"], list(meta["classes"])
    def idx(self, s): return np.where(self.split == s)[0]

store = Store(CACHE)
print("cache:", store.X.shape, "| leaf fraction of image: %.3f" % float(np.asarray(store.M[:min(len(store.M), 2000)]).mean()))

## 3. GPU data pipeline, augmentation and background randomisation (the innovation)

In [ ]:
def batches(store, index, bs, shuffle, rng, drop_last=False):
    order = rng.permutation(index) if shuffle else index
    chunks = [np.sort(order[i:i + bs]) for i in range(0, len(order), bs)]
    if drop_last and chunks and len(chunks[-1]) < bs: chunks = chunks[:-1]
    q = queue.Queue(maxsize=4)
    def producer():
        for b in chunks:
            x, m, y = torch.from_numpy(store.X[b]), torch.from_numpy(store.M[b]), torch.from_numpy(store.y[b])
            q.put((x.pin_memory() if dev == "cuda" else x, m, y))
        q.put(None)
    threading.Thread(target=producer, daemon=True).start()
    while True:
        it = q.get()
        if it is None: break
        x, m, y = it
        yield (x.to(dev, non_blocking=True).permute(0, 3, 1, 2).float() / 255,
               m.to(dev)[:, None].float(), y.to(dev).long())

def random_background(B, size, g):
    """Per-sample random background in [0,1]: solid colour / smooth noise texture / two-colour gradient."""
    r = lambda *s: torch.rand(*s, device=dev, generator=g)
    kind = torch.randint(0, 3, (B, 1, 1, 1), device=dev, generator=g)
    solid = r(B, 3, 1, 1).expand(B, 3, size, size)
    low = torch.randn(B, 3, 6, 6, device=dev, generator=g)
    tex = F.interpolate(low, size=size, mode="bilinear", align_corners=False)
    tex = (0.5 + 0.25 * tex) * r(B, 3, 1, 1) + 0.2 * r(B, 1, 1, 1)
    c1, c2 = r(B, 3, 1, 1), r(B, 3, 1, 1)
    ramp = torch.linspace(0, 1, size, device=dev)
    horiz = r(B, 1, 1, 1) < 0.5
    t = torch.where(horiz, ramp.view(1, 1, 1, size).expand(B, 1, size, size), ramp.view(1, 1, size, 1).expand(B, 1, size, size))
    grad = c1 * (1 - t) + c2 * t
    return torch.where(kind == 0, solid, torch.where(kind == 1, tex, grad)).clamp(0, 1)

def swap_background(x, m, g, p=1.0):
    """Replace background (mask == 0) by a random one for a fraction p of the batch; leaf pixels are untouched."""
    swapped = x * m + random_background(x.shape[0], x.shape[-1], g) * (1 - m)
    if p >= 1.0: return swapped
    sel = torch.rand(x.shape[0], 1, 1, 1, device=dev, generator=g) < p
    return torch.where(sel, swapped, x)

def gpu_augment(x, m, g, bg=False, p_bg=0.5):
    B = x.shape[0]
    k = torch.randint(0, 4, (B,), device=dev, generator=g)
    for r_ in (1, 2, 3):
        s = k == r_
        if s.any(): x[s], m[s] = torch.rot90(x[s], r_, (2, 3)), torch.rot90(m[s], r_, (2, 3))
    f = torch.rand(B, device=dev, generator=g) < 0.5
    if f.any(): x[f], m[f] = x[f].flip(3), m[f].flip(3)
    x = (x * (0.8 + 0.4 * torch.rand(B, 1, 1, 1, device=dev, generator=g))).clamp(0, 1)
    if bg: x = swap_background(x, m, g, p_bg)
    return x, m

# quick visual check of the augmentation (saved to OUT/figures_aug.png)
_g = torch.Generator(device=dev); _g.manual_seed(0)
_x, _m, _y = next(batches(store, store.idx("train")[:64], 8, False, None))
_a, _ = gpu_augment(_x.clone(), _m.clone(), _g, bg=True, p_bg=1.0)
fig, ax = plt.subplots(2, 8, figsize=(16, 4))
for j in range(8):
    ax[0, j].imshow(_x[j].permute(1, 2, 0).cpu()); ax[1, j].imshow(_a[j].permute(1, 2, 0).cpu()); ax[0, j].axis("off"); ax[1, j].axis("off")
ax[0, 0].set_title("original", fontsize=8); ax[1, 0].set_title("augmented + random background", fontsize=8)
fig.savefig(f"{OUT}/figures_aug.png", dpi=110); plt.close(fig)

## 4. Models, metrics, Grad-CAM

In [ ]:
def build(name, pretrained=None):
    kw = {"img_size": CFG["img"]} if name.startswith("vit") else {}
    return timm.create_model(name, pretrained=CFG["pretrained"] if pretrained is None else pretrained, num_classes=NC, **kw)

def cam_layer(model, name):
    if name.startswith("vit"): return None
    if hasattr(model, "layer4"): return model.layer4
    if hasattr(model, "stages"): return model.stages[-1]
    if hasattr(model, "blocks"): return model.blocks[-1]
    return None

def norm_stats(model):
    c = timm.data.resolve_data_config({}, model=model)
    return (torch.tensor(c["mean"], device=dev).view(1, 3, 1, 1), torch.tensor(c["std"], device=dev).view(1, 3, 1, 1))

def n_params(model): return sum(p.numel() for p in model.parameters())

def ece_np(P, y, bins=15):
    conf, pred = P.max(1), P.argmax(1)
    acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any(): e += s.mean() * abs(acc[s].mean() - conf[s].mean())
    return float(e)

def full_metrics(P, y):
    P = P.astype(np.float64); P = P / P.sum(1, keepdims=True)
    p = P.argmax(1)
    cm = confusion_matrix(y, p, labels=range(NC))
    pm, rm, fm, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y, p, average="weighted", zero_division=0)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
    spec = tn / np.maximum(tn + fp, 1)
    try: auc = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(NC)))
    except ValueError: auc = float("nan")
    return dict(accuracy=float((p == y).mean()), precision_macro=pm, recall_macro=rm, f1_macro=fm, f1_weighted=fw,
                specificity_macro=float(spec.mean()), roc_auc_macro_ovr=float(auc), ece=ece_np(P, y)), cm, spec

@torch.no_grad()
def predict(model, mean, std, index, swap=False):
    model.eval(); P, Y = [], []
    g = torch.Generator(device=dev); g.manual_seed(123)          # identical swapped backgrounds for every model
    for x, m, y in batches(store, index, 128, False, None):
        if swap: x = swap_background(x, m, g, 1.0)
        with torch.autocast(dev, dtype=torch.float16, enabled=(dev == "cuda")):
            out = model((x - mean) / std)
        P.append(out.float().softmax(1).cpu()); Y.append(y.cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def grad_cam(model, layer, x):
    acts, grads = {}, {}
    def fwd(_, __, out):
        acts["a"] = out; out.register_hook(lambda g_: grads.__setitem__("g", g_))
    h = layer.register_forward_hook(fwd)
    model.eval(); x = x.clone().requires_grad_(True)
    logits = model(x); model.zero_grad()
    logits.gather(1, logits.argmax(1, keepdim=True)).sum().backward()
    h.remove()
    w = grads["g"].mean((2, 3), keepdim=True)
    cam = F.relu((w * acts["a"]).sum(1, keepdim=True))
    return F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0].detach(), logits.detach()

def leaf_focus(model, layer, mean, std, index, swap, n):
    """Fraction of Grad-CAM mass that falls inside the leaf (clean or background-swapped test images)."""
    sel = np.sort(np.random.RandomState(0).choice(index, min(n, len(index)), replace=False))
    g = torch.Generator(device=dev); g.manual_seed(7); vals = []
    for x, m, _ in batches(store, sel, 50, False, None):
        if swap: x = swap_background(x, m, g, 1.0)
        cam, _ = grad_cam(model, layer, (x - mean) / std)
        tot = cam.flatten(1).sum(1); inside = (cam * m[:, 0]).flatten(1).sum(1)
        vals.append(torch.where(tot > 1e-8, inside / tot.clamp_min(1e-8), torch.full_like(tot, float("nan"))).cpu())
    return float(torch.nanmean(torch.cat(vals)))

## 5. Training and evaluation of one run

In [ ]:
def train_run(name, bg, seed, tag):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    g = torch.Generator(device=dev); g.manual_seed(seed)
    rng = np.random.RandomState(seed)
    model = build(name).to(dev)
    if dev == "cuda": model = model.to(memory_format=torch.channels_last)
    mean, std = norm_stats(model)
    tr_i, va_i = store.idx("train"), store.idx("val")
    bs, E = CFG["batch"], CFG["epochs"]
    steps = len(tr_i) // bs
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], epochs=E, steps_per_epoch=steps, pct_start=0.2)
    try: scaler = torch.amp.GradScaler(dev, enabled=(dev == "cuda"))
    except TypeError: scaler = torch.cuda.amp.GradScaler(enabled=(dev == "cuda"))
    crit = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    hist, best_acc, best_ep, t0 = [], -1.0, 0, time.time()
    for ep in range(1, E + 1):
        model.train(); tl = tc = n = 0
        for x, m, y in batches(store, tr_i, bs, True, rng, drop_last=True):
            x, m = gpu_augment(x, m, g, bg, CFG["p_bg"])
            with torch.autocast(dev, dtype=torch.float16, enabled=(dev == "cuda")):
                out = model((x - mean) / std); loss = crit(out.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); n += len(y)
        P, Y = predict(model, mean, std, va_i)
        va_acc = float((P.argmax(1) == Y).mean()); va_loss = float(-np.log(P[np.arange(len(Y)), Y] + 1e-9).mean())
        hist.append(dict(epoch=ep, train_loss=tl / n, train_acc=tc / n, val_loss=va_loss, val_acc=va_acc, elapsed_s=time.time() - t0))
        print(f"[{tag}] ep {ep:02d} train {tl/n:.3f}/{tc/n:.3f} | val {va_loss:.3f}/{va_acc:.4f} | {time.time()-t0:.0f}s", flush=True)
        if va_acc > best_acc:                                   # checkpoint chosen on VALIDATION accuracy only
            best_acc, best_ep = va_acc, ep
            torch.save(model.state_dict(), f"{OUT}/ckpt/{tag}.pt")
    train_s = time.time() - t0
    model.load_state_dict(torch.load(f"{OUT}/ckpt/{tag}.pt", map_location=dev))
    pd.DataFrame(hist).to_csv(f"{OUT}/runs/history_{tag}.csv", index=False)

    # ---- test-set evaluation (touched once per run)
    te_i = store.idx("test")
    P, Y = predict(model, mean, std, te_i)
    met, cm, spec = full_metrics(P, Y)
    Ps, Ys = predict(model, mean, std, te_i, swap=True)
    ps = Ps.argmax(1)
    met["bgswap_accuracy"] = float((ps == Ys).mean())
    met["bgswap_f1_macro"] = float(precision_recall_fscore_support(Ys, ps, average="macro", zero_division=0)[2])
    layer = cam_layer(model, name)
    if layer is not None:
        met["leaf_focus_clean"] = leaf_focus(model, layer, mean, std, te_i, False, CFG["n_xai"])
        met["leaf_focus_bgswap"] = leaf_focus(model, layer, mean, std, te_i, True, CFG["n_xai"])
    xb = torch.randn(1, 3, S, S, device=dev); model.eval()
    with torch.no_grad():
        for _ in range(5): model(xb)
        if dev == "cuda": torch.cuda.synchronize()
        t1 = time.time()
        for _ in range(30): model(xb)
        if dev == "cuda": torch.cuda.synchronize()
    met.update(tag=tag, model=name, bg_aug=bool(bg), seed=seed, params=n_params(model), best_epoch=best_ep,
               train_seconds=train_s, latency_ms_batch1=(time.time() - t1) / 30 * 1000, test_images=int(len(Y)))
    json.dump(met, open(f"{OUT}/runs/metrics_{tag}.json", "w"), indent=2)
    pd.DataFrame(cm, index=store.classes, columns=store.classes).to_csv(f"{OUT}/runs/cm_{tag}.csv")
    rep = pd.DataFrame(classification_report(Y, P.argmax(1), labels=range(NC), target_names=store.classes, output_dict=True, zero_division=0)).T
    rep["specificity"] = list(spec) + [np.nan] * 3
    rep.to_csv(f"{OUT}/runs/perclass_{tag}.csv")
    del model, opt
    if dev == "cuda": torch.cuda.empty_cache()
    return met

def run_one(name, bg, seed):
    tag = f"{name}_{'bg' if bg else 'plain'}_s{seed}"
    f = f"{OUT}/runs/metrics_{tag}.json"
    if os.path.exists(f):                                       # resumable after a disconnect
        print("skip (done):", tag); return json.load(open(f))
    return train_run(name, bg, seed, tag)

## 6. Run all experiments
* **CO4 comparison:** every model in `compare_models`, plain fine-tuning, seed 0, identical split.
* **CO5 ablation:** `main_model`, plain vs background-randomisation training, seeds in `CFG["seeds"]`.

In [ ]:
runs = [(m, False, 0) for m in CFG["compare_models"]]
for s in CFG["seeds"]:
    runs += [(CFG["main_model"], False, s), (CFG["main_model"], True, s)]
seen, plan = set(), []
for r in runs:
    if r not in seen: seen.add(r); plan.append(r)
print(len(plan), "runs:", plan)
results = []
for name, bg, seed in plan:
    results.append(run_one(name, bg, seed))
    pd.DataFrame(results).to_csv(f"{OUT}/results_all_runs.csv", index=False)

## 7. Tables for the report (CO3 / CO4 / CO5)

In [ ]:
df = pd.DataFrame([json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/runs/metrics_*.json"))])
df.to_csv(f"{OUT}/results_all_runs.csv", index=False)

def ms(x, d=4):
    x = pd.Series(x).dropna()
    if len(x) == 0: return ""
    return f"{x.mean():.{d}f} ± {x.std(ddof=0):.{d}f}" if len(x) > 1 else f"{x.iloc[0]:.{d}f}"

# CO4: all models, plain fine-tuning (main model = mean ± std over its seeds)
rows = []
for name in CFG["compare_models"]:
    d = df[(df.model == name) & (~df.bg_aug)]
    if d.empty: continue
    rows.append({"Model": name, "seeds": len(d), "Accuracy": ms(d.accuracy), "Precision(macro)": ms(d.precision_macro),
                 "Recall(macro)": ms(d.recall_macro), "F1(macro)": ms(d.f1_macro), "Specificity": ms(d.specificity_macro),
                 "AUC(macro OvR)": ms(d.roc_auc_macro_ovr, 5), "ECE": ms(d.ece), "BG-swap Acc": ms(d.bgswap_accuracy),
                 "Params(M)": round(d.params.iloc[0] / 1e6, 2), "ms/img (batch 1)": ms(d.latency_ms_batch1, 2)})
comparison = pd.DataFrame(rows); comparison.to_csv(f"{OUT}/comparison.csv", index=False)

# CO5: main model, plain vs background randomisation (mean ± std over seeds)
rows = []
for bg, label in [(False, "Plain fine-tuning"), (True, "+ background randomisation (proposed)")]:
    d = df[(df.model == CFG["main_model"]) & (df.bg_aug == bg)]
    if d.empty: continue
    rows.append({"Experiment": label, "seeds": len(d), "Accuracy": ms(d.accuracy), "F1(macro)": ms(d.f1_macro),
                 "AUC": ms(d.roc_auc_macro_ovr, 5), "ECE": ms(d.ece), "BG-swap Acc": ms(d.bgswap_accuracy),
                 "BG-swap F1": ms(d.bgswap_f1_macro),
                 "Leaf-focus clean": ms(d.get("leaf_focus_clean", pd.Series(dtype=float))),
                 "Leaf-focus BG-swap": ms(d.get("leaf_focus_bgswap", pd.Series(dtype=float)))})
ablation = pd.DataFrame(rows); ablation.to_csv(f"{OUT}/ablation.csv", index=False)

# Published results for context. ADD your own references with the protocol each paper used. Only entries you can cite belong here.
PUBLISHED = pd.DataFrame([
    {"Reference": "Mohanty et al., Front. Plant Sci. 2016 (GoogLeNet, transfer learning)", "Accuracy": "~99.35 % (as reported)",
     "Protocol": "full 38-class colour set, 80/20 split, 256x256 - NOT identical to this notebook; not reproduced here"},
])
PUBLISHED.to_csv(f"{OUT}/published_reference.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("== CO4 comparison (same split, same preprocessing) =="); print(comparison.to_string(index=False))
print("\n== CO5 ablation =="); print(ablation.to_string(index=False))
print("\n== Published (context only) =="); print(PUBLISHED.to_string(index=False))

## 8. Figures: curves, confusion matrix, Grad-CAM

In [ ]:
main = CFG["main_model"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for bg, c in [(False, "tab:gray"), (True, "tab:red")]:
    f = f"{OUT}/runs/history_{main}_{'bg' if bg else 'plain'}_s{CFG['seeds'][0]}.csv"
    if os.path.exists(f):
        h = pd.read_csv(f); lab = "bg-randomisation" if bg else "plain"
        ax[0].plot(h.epoch, h.train_loss, "--", c=c, label=f"{lab} train"); ax[0].plot(h.epoch, h.val_loss, "-", c=c, label=f"{lab} val")
        ax[1].plot(h.epoch, h.train_acc, "--", c=c, label=f"{lab} train"); ax[1].plot(h.epoch, h.val_acc, "-", c=c, label=f"{lab} val")
ax[0].set_title("Loss"); ax[1].set_title("Accuracy")
for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=7); a.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f"{OUT}/training_curve.png", dpi=150); plt.close(fig)

for bg in [False, True]:
    f = f"{OUT}/runs/cm_{main}_{'bg' if bg else 'plain'}_s{CFG['seeds'][0]}.csv"
    if not os.path.exists(f): continue
    cm = pd.read_csv(f, index_col=0).values; cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(11, 10)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    short = [c.replace("___", ": ").replace("_", " ")[:28] for c in store.classes]
    ax.set_xticks(range(NC)); ax.set_yticks(range(NC)); ax.set_xticklabels(short, rotation=90, fontsize=5); ax.set_yticklabels(short, fontsize=5)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"Confusion matrix (test) - {main} {'+bg-rand' if bg else 'plain'}")
    fig.colorbar(im, fraction=0.03); fig.tight_layout(); fig.savefig(f"{OUT}/confusion_{main}_{'bg' if bg else 'plain'}.png", dpi=160); plt.close(fig)

# Grad-CAM: plain vs background-randomised model, clean vs background-swapped test images
rs = np.random.RandomState(3); pick = np.sort(rs.choice(store.idx("test"), 6, replace=False))
xs, ms_, ys = next(batches(store, pick, 6, False, None))
xsw = swap_background(xs, ms_, torch.Generator(device=dev).manual_seed(7), 1.0)
panels = []
for bg in [False, True]:
    ck = f"{OUT}/ckpt/{main}_{'bg' if bg else 'plain'}_s{CFG['seeds'][0]}.pt"
    if not os.path.exists(ck) or cam_layer(build(main, False), main) is None: continue
    mdl = build(main, False).to(dev); mdl.load_state_dict(torch.load(ck, map_location=dev)); mean, std = norm_stats(mdl)
    for lab, imgs in [("clean", xs), ("BG-swap", xsw)]:
        cam, lg = grad_cam(mdl, cam_layer(mdl, main), (imgs - mean) / std); panels.append((f"{'bg-rand' if bg else 'plain'}\n{lab}", imgs, cam, lg))
if panels:
    fig, axs = plt.subplots(len(panels), 6, figsize=(12, 2 * len(panels)))
    for r_, (lab, imgs, cam, lg) in enumerate(panels):
        for j in range(6):
            c = cam[j] / cam[j].max().clamp_min(1e-8)
            axs[r_, j].imshow(imgs[j].permute(1, 2, 0).cpu()); axs[r_, j].imshow(c.cpu(), cmap="jet", alpha=0.45)
            axs[r_, j].set_xlabel("correct" if lg[j].argmax().item() == int(ys[j]) else "wrong", fontsize=6)
            axs[r_, j].set_xticks([]); axs[r_, j].set_yticks([])
        axs[r_, 0].set_ylabel(lab, fontsize=7)
    fig.tight_layout(); fig.savefig(f"{OUT}/gradcam_examples.png", dpi=150); plt.close(fig)
print("figures saved in", OUT)

## 9. Package the results (download for your submission)

In [ ]:
zip_path = shutil.make_archive(os.path.join(WORK, "plantvillage_results"), "zip", OUT)
print("zipped ->", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("(download skipped:", type(e).__name__, ")")